# Practice 3 — Security, Testing, and Performance

Safe parsing, secrets, subprocess safety, transactions, algorithms.

**How it works:** each exercise is almost-finished code with a few `___` blanks. Fill them in, run the cell, then run the `check(...)` cell below it. ✅ means move on. ❌ shows your result and the test input, never the answer, and failing twice unlocks a hint.

Prefer a terminal? The same exercises are in [`practice/exercises/advanced_03_security_testing_performance.py`](../../practice/exercises/advanced_03_security_testing_performance.py).

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "exercises"))
    sys.path.insert(0, os.getcwd())

## Setup

Run this cell first.

In [ ]:
from __future__ import annotations
from check import Checker, ___  # ___ marks the blanks you fill in

check = Checker('advanced_03_security_testing_performance')

import bisect
import heapq
import hmac
import itertools
import json
import re
import secrets
import sqlite3
from collections.abc import Sequence

SENSITIVE_KEYS = {"password", "token", "secret", "api_key", "authorization"}

## Exercises

### 1. `parse_limited_json`

Parse a JSON object safely.

In [ ]:
def parse_limited_json(text: str, max_bytes: int = 10_000) -> dict:
    """Parse a JSON object safely.

    Check the size (UTF-8 bytes) BEFORE parsing; raise ValueError if too big, if
    the JSON is invalid, or if the top level is not an object.
    """
    if ___:
        raise ValueError("payload too large")
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError("invalid JSON") from error
    if ___:
        raise ValueError("expected a JSON object")
    return data

In [ ]:
check('parse_limited_json')

### 2. `make_token`

Return a URL-safe random token of at least 32 bytes of entropy (use `secrets`, not `random`).

In [ ]:
def make_token() -> str:
    """Return a URL-safe random token of at least 32 bytes of entropy (use `secrets`, not `random`)."""
    return ___

In [ ]:
check('make_token')

### 3. `tokens_match`

Compare secrets in constant time with hmac.compare_digest.

In [ ]:
def tokens_match(expected: str, provided: str) -> bool:
    """Compare secrets in constant time with hmac.compare_digest."""
    return ___

In [ ]:
check('tokens_match')

### 4. `grep_command`

Build an argument list for `grep -n -- PATTERN PATH` (no shell, no string joining).

In [ ]:
def grep_command(pattern: str, path: str) -> list[str]:
    """Build an argument list for `grep -n -- PATTERN PATH` (no shell, no string joining).

    The '--' stops a pattern like '-rf' from being read as an option.
    """
    return ___

In [ ]:
check('grep_command')

### 5. `redact`

Return a copy with values of sensitive keys (case-insensitive, nested dicts too) replaced by '***'.

In [ ]:
def redact(record: dict) -> dict:
    """Return a copy with values of sensitive keys (case-insensitive, nested dicts too) replaced by '***'."""
    cleaned = {}
    for key, value in record.items():
        if ___:
            cleaned[key] = "***"
        elif ___:
            cleaned[key] = redact(value)
        else:
            cleaned[key] = value
    return cleaned

In [ ]:
check('redact')

### 6. `limit_text`

Return text unchanged if short enough; raise ValueError otherwise.

In [ ]:
def limit_text(text: str, max_chars: int = 1000) -> str:
    """Return text unchanged if short enough; raise ValueError otherwise."""
    if ___:
        raise ValueError(f"input longer than {max_chars} characters")
    return text

In [ ]:
check('limit_text')

### 7. `StablePriorityQueue`

Min-priority queue; items with equal priority come out in insertion (FIFO) order.

In [ ]:
class StablePriorityQueue:
    """Min-priority queue; items with equal priority come out in insertion (FIFO) order.

    Hint: push (priority, sequence_number, item) tuples onto a heapq list.
    """

    def __init__(self) -> None:
        self._heap: list = []
        self._counter = ___

    def push(self, priority: int, item) -> None:
        heapq.heappush(___, (priority, next(self._counter), item))

    def pop(self):
        if ___:
            raise IndexError("pop from empty queue")
        return heapq.heappop(self._heap)[2]

    def __len__(self) -> int:
        return ___

In [ ]:
check('StablePriorityQueue')

### 8. `find_index`

Binary search with bisect; return the index of target or -1 if absent.

In [ ]:
def find_index(sorted_values: Sequence, target) -> int:
    """Binary search with bisect; return the index of target or -1 if absent."""
    index = bisect.bisect_left(sorted_values, target)
    return ___

In [ ]:
check('find_index')

### 9. `add_users`

Insert all emails into users(email UNIQUE) in ONE transaction.

In [ ]:
def add_users(conn: sqlite3.Connection, emails: list[str]) -> None:
    """Insert all emails into users(email UNIQUE) in ONE transaction.

    If any insert fails (duplicate or invalid email), insert none of them and
    raise ValueError. A valid email matches  something@something.something
    """
    pattern = ___
    try:
        with conn:
            for email in emails:
                if ___:
                    raise ValueError(f"invalid email: {email!r}")
                conn.execute("INSERT INTO users (email) VALUES (?)", (email,))
    except sqlite3.IntegrityError as error:
        raise ValueError("duplicate email") from error

In [ ]:
check('add_users')

### 10. `hash_password`

PBKDF2-HMAC-SHA256 of the password, returned as hex. Hint: hashlib.pbkdf2_hmac.

*Checked together with a later exercise that uses it.*

In [ ]:
def hash_password(password: str, salt: bytes, iterations: int = 100_000) -> str:
    """PBKDF2-HMAC-SHA256 of the password, returned as hex. Hint: hashlib.pbkdf2_hmac."""
    import hashlib
    return ___

### 11. `verify_password`

Recompute the hash and compare in constant time.

In [ ]:
def verify_password(password: str, salt: bytes, expected_hex: str, iterations: int = 100_000) -> bool:
    """Recompute the hash and compare in constant time."""
    return ___

In [ ]:
check('verify_password')

### 12. `safe_join`

Join a user-supplied relative path under base and return the resolved absolute path.

In [ ]:
def safe_join(base: str, user_path: str) -> str:
    """Join a user-supplied relative path under base and return the resolved absolute path.

    Raise ValueError if the result escapes base ('../etc/passwd', absolute paths, ...).
    """
    from pathlib import Path
    root = Path(base).resolve()
    target = (root / user_path).resolve()
    if ___:
        raise ValueError("path escapes the base directory")
    return ___

In [ ]:
check('safe_join')

### 13. `merge_sorted`

Merge already-sorted sequences into one sorted list lazily. Hint: heapq.merge.

In [ ]:
def merge_sorted(*sequences: Sequence) -> list:
    """Merge already-sorted sequences into one sorted list lazily. Hint: heapq.merge."""
    return ___

In [ ]:
check('merge_sorted')

### 14. `top_k`

The k largest values, largest first, in O(n log k). Hint: heapq.nlargest.

In [ ]:
def top_k(values: Sequence[float], k: int) -> list:
    """The k largest values, largest first, in O(n log k). Hint: heapq.nlargest."""
    return ___

In [ ]:
check('top_k')

## Finished?

In [ ]:
check.summary()